# Register an MLTable Data Asset

Create MLTable metadata around a taxi CSV, register an immutable Azure ML data asset, and retrieve it for verification.

**Sources:** Adapted from [AzureML-deep-dive-L200](https://github.com/jomedinagomez/AzureML-deep-dive-L200/blob/efa77413408a8053781096c41fb07cb882c8f518/taxi-fare-predictions/notebooks/dataset_registration.ipynb) and [Azure/azureml-examples](https://github.com/Azure/azureml-examples/tree/main/sdk/python/assets/data). Both sources use the MIT License; details are in `workshop/SOURCES.md`.

In [23]:
from pathlib import Path
import os
import shutil

import mltable
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
DATA_NAME = os.environ["WORKSHOP_DATA_NAME"]
REGISTER = os.getenv("REGISTER_FOUNDATION_DATA", "false").lower() in {"1", "true", "yes"}

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [24]:
source_csv = WORKSHOP_ROOT / "data/taxi/raw/yellowTaxiData.csv"
generated_mltable = WORKSHOP_ROOT / "outputs/foundation_mltable"
if generated_mltable.exists():
    shutil.rmtree(generated_mltable)

table = mltable.from_delimited_files(paths=[{"file": str(source_csv)}])
table.save(str(generated_mltable), colocated=True)

data_definition = Data(
    name=DATA_NAME,
    type=AssetTypes.MLTABLE,
    path=str(generated_mltable),
    description="Taxi data registered during the Azure ML workshop",
    tags={"workshop": "azureml-h2o", "source": "taxi-sample"},
)

if REGISTER:
    registered_data = ml_client.data.create_or_update(data_definition)
    verified_data = ml_client.data.get(
        DATA_NAME,
        version=registered_data.version,
    )
    assert verified_data.name == DATA_NAME
    assert verified_data.version == registered_data.version
    print(f"Registered data: {verified_data.name}:{verified_data.version}")
else:
    print(f"Prepared data definition: {DATA_NAME} (version assigned on registration)")
    print("Registration disabled. Set REGISTER_FOUNDATION_DATA=true in workshop/.env.")

Failed to download MLTable metadata jsonschema from "https://azuremlschemasprod.azureedge.net/latest/MLTable.schema.json", skipping validation


Registered data: workshop-taxi-data:3


## Expected Result

A local MLTable folder is prepared. When enabled, Azure ML assigns an immutable data version and the notebook retrieves it successfully.

Next: `03_register_environment.ipynb`.